# 🪰 Musca domestica Cohort Simulation — Colab Launcher

This notebook clones the app from GitHub, installs dependencies, and opens
a public tunnel so you can use the full Streamlit interface from inside
Google Colab.

**How to use:**
1. Run **Cell 1** once to install packages (takes ~60 seconds)
2. Paste your free [ngrok](https://ngrok.com) auth token into **Cell 2** and run it
3. Run **Cell 3** — a public URL will appear; click it to open the app
4. When finished, run **Cell 4** to shut everything down cleanly

> **Get a free ngrok token:** Sign up at https://ngrok.com (free, no credit card),
> then go to *Dashboard → Your Authtoken* and copy the token string.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Clone repo and install dependencies
# Run once per Colab session.  Output is suppressed; ✅ means success.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os

GITHUB_REPO = 'https://github.com/ackuyucu/Ekoloji_Lab.git'
APP_DIR     = 'Ekoloji_Lab'

# Clone (skip if already present)
if not os.path.isdir(APP_DIR):
    print('Cloning repository…')
    subprocess.run(['git', 'clone', GITHUB_REPO, APP_DIR], check=True)
else:
    print('Repository already cloned — pulling latest…')
    subprocess.run(['git', '-C', APP_DIR, 'pull'], check=True)

# Install Python dependencies quietly
print('Installing dependencies…')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'streamlit', 'pyngrok', 'numpy', 'pandas',
     'matplotlib', 'openpyxl'],
    check=True
)

print('✅  Ready.  Proceed to Cell 2.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Configure ngrok tunnel
# Paste your free token from https://dashboard.ngrok.com/get-started/your-authtoken
# ─────────────────────────────────────────────────────────────────────────────
from pyngrok import ngrok, conf

NGROK_TOKEN = 'PASTE_YOUR_TOKEN_HERE'   # ← replace with your token

ngrok.set_auth_token(NGROK_TOKEN)
print('✅  ngrok token configured.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — Launch the Streamlit app and open a public tunnel
# After running this cell, click the URL that appears to open the app.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, time, os
from pyngrok import ngrok
from IPython.display import display, HTML

PORT     = 8501
LOG_FILE = '/tmp/streamlit.log'
APP_PATH = os.path.join(APP_DIR, 'musca_streamlit_app', 'app.py')

# Kill any previous Streamlit processes
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(1)

# Close previous ngrok tunnels
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

# Start Streamlit in the background
with open(LOG_FILE, 'w') as log:
    proc = subprocess.Popen(
        [
            'streamlit', 'run', APP_PATH,
            f'--server.port={PORT}',
            '--server.headless=true',
            '--server.enableCORS=false',
            '--server.enableXsrfProtection=false',
        ],
        stdout=log, stderr=log
    )

# Wait for Streamlit to be ready
print('Starting Streamlit', end='')
for _ in range(20):
    time.sleep(1)
    with open(LOG_FILE) as f:
        if 'You can now view' in f.read():
            break
    print('.', end='', flush=True)
print()

# Open ngrok tunnel
tunnel     = ngrok.connect(PORT)
public_url = tunnel.public_url

display(HTML(f"""
<div style="background:#f0fdf4; border:2px solid #16a34a; border-radius:10px;
             padding:1rem 1.5rem; font-family:sans-serif; margin-top:0.5rem">
  <h3 style="color:#15803d; margin:0 0 0.4rem 0">✅  App is running!</h3>
  <p style="margin:0; font-size:1rem">
    Click to open:
    <a href="{public_url}" target="_blank"
       style="color:#1d4ed8; font-weight:700">{public_url}</a>
  </p>
  <p style="margin:0.4rem 0 0 0; color:#555; font-size:0.85rem">
    The link stays active as long as this Colab session is running.
  </p>
</div>
"""))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — (Optional) Shut down the app and close the tunnel
# ─────────────────────────────────────────────────────────────────────────────
import subprocess
from pyngrok import ngrok

ngrok.kill()
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
print('🛑  Streamlit stopped and tunnel closed.')

---

## Troubleshooting

**The URL opens but shows a blank page:**  
Wait 10 more seconds and refresh. Streamlit sometimes needs extra time on
first load.

**`ERR_NGROK_3200` — tunnel expired:**  
Free ngrok tunnels expire after ~2 hours. Re-run Cell 3 to get a fresh URL.

**`ModuleNotFoundError: No module named 'streamlit'`:**  
Re-run Cell 1. Colab occasionally loses installed packages after a
runtime restart.

**I don't want to use ngrok:**  
Replace Cells 2 and 3 with the `localtunnel` alternative below:

```python
# localtunnel alternative (no account needed)
import subprocess, time
from IPython.display import display, HTML

subprocess.Popen(['streamlit', 'run', 'musca-sim/app.py',
                  '--server.port=8501', '--server.headless=true'],
                 stdout=open('/tmp/st.log','w'), stderr=subprocess.STDOUT)
time.sleep(4)

proc = subprocess.run(['npx', '--yes', 'localtunnel',
                       '--port', '8501'],
                      capture_output=True, text=True, timeout=15)
url = [l for l in proc.stdout.splitlines() if 'https://' in l][0].split()[-1]
display(HTML(f'<a href="{url}" target="_blank">Open app → {url}</a>'))
```